In [1]:
# Imports
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# Preprocess
data_frame = pd.read_csv('../data/all_fights.csv')

# drop weird stances
data_frame = data_frame[~data_frame["blue_stance"].isin(["Switch ", "Open Stance", "Unknown"])]
data_frame = data_frame[~data_frame["red_stance"].isin(["Switch ", "Open Stance", "Unknown"])]

# dropping reach diff over 40 (1 entry)
data_frame = data_frame[(data_frame["reach_diff"] > -40) | (data_frame["reach_diff"] < 40)]

# dropping Catch Weight from weight_class
data_frame = data_frame[~data_frame["weight_class"].isin(["Catch Weight"])]

# Round_diff - check this one a bit better, only 1 super outlier
data_frame = data_frame[(data_frame["rounds_diff"] > -100) | (data_frame["rounds_diff"] < 100)]
#print(data_frame["weight_class"].value_counts())

In [3]:
diff_features = data_frame.filter(regex=r'_diff').columns.to_list()
diff_features_no_odds = [f for f in diff_features if f != 'odds_diff']

extra_features = ['gender', 'weight_class', 'red_stance', 'blue_stance']

feature_set_with_odds= diff_features + extra_features
feature_set_no_odds= diff_features_no_odds + extra_features

y = (data_frame['red_winner'] == 't').astype(int)

DATE = "2023-06-01"

train_mask = data_frame['fight_date'] <= DATE
test_mask = data_frame['fight_date'] > DATE

y_train = y[train_mask]
y_test = y[test_mask]

X_train_odds = data_frame.loc[train_mask, feature_set_with_odds]
X_test_odds = data_frame.loc[test_mask, feature_set_with_odds]

X_train_no_odds = data_frame.loc[train_mask, feature_set_no_odds]
X_test_no_odds = data_frame.loc[test_mask, feature_set_no_odds]

print('=== With Odds ===')
print(X_train_odds.shape)
print(X_test_odds.shape)

print('=== Without Odds ===')
print(X_train_no_odds.shape)
print(X_test_no_odds.shape)

print("Mean")
print(y_train.mean())

=== With Odds ===
(5668, 21)
(1515, 21)
=== Without Odds ===
(5668, 20)
(1515, 20)
Mean
0.5827452364149612


In [4]:
# Logistic Regression
# Odds included
X_test_dummy_odds = pd.get_dummies(X_test_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_train_dummy_odds = pd.get_dummies(X_train_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train_dummy_odds, y_train)
preds_odds = lr.predict(X_test_dummy_odds)
print('=== With Odds Diff ===')
print(accuracy_score(y_test, preds_odds))
print(classification_report(y_test, preds_odds))
# Odds not included
X_test_dummy_no_odds = pd.get_dummies(X_test_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_train_dummy_no_odds = pd.get_dummies(X_train_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train_dummy_no_odds, y_train)
preds_no_odds = lr.predict(X_test_dummy_no_odds)
print('=== Without Odds Diff ===')
print(accuracy_score(y_test, preds_no_odds))
print(classification_report(y_test, preds_no_odds))

=== With Odds Diff ===
0.7023102310231023
              precision    recall  f1-score   support

           0       0.69      0.60      0.64       673
           1       0.71      0.79      0.75       842

    accuracy                           0.70      1515
   macro avg       0.70      0.69      0.69      1515
weighted avg       0.70      0.70      0.70      1515

=== Without Odds Diff ===
0.6059405940594059
              precision    recall  f1-score   support

           0       0.59      0.36      0.45       673
           1       0.61      0.80      0.69       842

    accuracy                           0.61      1515
   macro avg       0.60      0.58      0.57      1515
weighted avg       0.60      0.61      0.58      1515



In [5]:
# Random Forest
X_test_dummy_odds = pd.get_dummies(X_test_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_train_dummy_odds = pd.get_dummies(X_train_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])

rfc = RandomForestClassifier(random_state=26)
rfc.fit(X_train_dummy_odds, y_train)
pred_odds = rfc.predict(X_test_dummy_odds)

print('Random Forest')
print('=== With Odds Diff ===')
print(accuracy_score(y_test, pred_odds))
print(classification_report(y_test, pred_odds))

X_test_dummy_no_odds = pd.get_dummies(X_test_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_train_dummy_no_odds = pd.get_dummies(X_train_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])

rfc.fit(X_train_dummy_no_odds, y_train)
pred_no_odds = rfc.predict(X_test_dummy_no_odds)

print('=== Without Odds Diff ===')
print(accuracy_score(y_test, pred_no_odds))
print(classification_report(y_test, pred_no_odds))


Random Forest
=== With Odds Diff ===
0.6759075907590759
              precision    recall  f1-score   support

           0       0.66      0.56      0.61       673
           1       0.69      0.77      0.72       842

    accuracy                           0.68      1515
   macro avg       0.67      0.66      0.67      1515
weighted avg       0.67      0.68      0.67      1515

=== Without Odds Diff ===
0.5887788778877888
              precision    recall  f1-score   support

           0       0.56      0.37      0.44       673
           1       0.60      0.77      0.67       842

    accuracy                           0.59      1515
   macro avg       0.58      0.57      0.56      1515
weighted avg       0.58      0.59      0.57      1515



In [ ]:
#Tuned Random Forest - No odds
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators' : [100, 200, 500],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 5, 10, 20]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=26),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)
grid_search.fit(X_train_dummy_no_odds, y_train)

print('=== Best params ===')
print(grid_search.best_params_)
print('=== Best score ===')
print(grid_search.best_score_)

best_rf = grid_search.best_estimator_
preds = best_rf.predict(X_test_dummy_no_odds)
print('=== Scores ===')
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

print('=== Importances ===')
importances = pd.Series(best_rf.feature_importances_, index=X_train_dummy_no_odds.columns)
print(importances.sort_values(ascending=False).head(10))

Fitting 5 folds for each of 48 candidates, totalling 240 fits
=== Best params ===
{'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 200}
=== Best score ===
0.6088538334493027
=== Scores ===
0.6072607260726073
              precision    recall  f1-score   support

           0       0.63      0.29      0.39       673
           1       0.60      0.86      0.71       842

    accuracy                           0.61      1515
   macro avg       0.61      0.57      0.55      1515
weighted avg       0.61      0.61      0.57      1515

=== Importances ====
age_diff           0.153834
td_diff            0.119382
sig_str_diff       0.102022
sub_att_diff       0.068870
rounds_diff        0.066037
losses_diff        0.065738
reach_diff         0.050140
win_streak_diff    0.042292
height_diff        0.040924
wins_diff          0.037856
dtype: float64


In [8]:
#Tuned Random Forest - Odds included
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators' : [100, 200, 500],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 5, 10, 20]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=26),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)
grid_search.fit(X_train_dummy_odds, y_train)

print('=== Best params ===')
print(grid_search.best_params_)
print('=== Best score ===')
print(grid_search.best_score_)

best_rf = grid_search.best_estimator_
preds = best_rf.predict(X_test_dummy_odds)
print('=== Scores ===')
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

print('=== With Odds ===')
importances = pd.Series(best_rf.feature_importances_, index=X_train_dummy_odds.columns)
print(importances.sort_values(ascending=False).head(10))

Fitting 5 folds for each of 48 candidates, totalling 240 fits
=== Best params ===
{'max_depth': None, 'min_samples_leaf': 20, 'n_estimators': 200}
=== Best score ===
0.6561371147131665
=== Scores ===
0.689108910891089
              precision    recall  f1-score   support

           0       0.68      0.56      0.62       673
           1       0.69      0.79      0.74       842

    accuracy                           0.69      1515
   macro avg       0.69      0.68      0.68      1515
weighted avg       0.69      0.69      0.68      1515

=== With Odds ===
odds_diff       0.418078
age_diff        0.082928
td_diff         0.072049
sig_str_diff    0.070986
sub_att_diff    0.043821
rounds_diff     0.040179
losses_diff     0.035555
reach_diff      0.034351
height_diff     0.025467
wins_diff       0.023992
dtype: float64


In [ ]:
print(grid_search.best_estimator_)